# Simple Neural Network With PyTorch Lightning

In this lab, we'll see how to use PyTorch Lightning to build simple neural networks.

You don't need a GPU from Colab for this lab: everything is fast on CPU. You can therefore switch environments if the one you were given has a GPU (`Runtime > Change runtime type`).

In [ ]:
!pip install -q lightning torchmetrics optuna tensorboard
import datetime

import lightning
import optuna
import torch
import torchmetrics
import torchvision
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

## Fetching the Data

This time, we'll get the dataset through torchvision.

In [ ]:
train_data = torchvision.datasets.MNIST("data", train=True, download=True)
test_data = torchvision.datasets.MNIST("data", train=False, download=True)
X_train, y_train = train_data.data, train_data.targets
X_test, y_test = test_data.data, test_data.targets

## Let's Look at the Data

In the next cells, study how the data is stored.

In [ ]:
# Your code here

### Solution

In [ ]:
print(f"X_train shape:  {tuple(X_train.shape)}")
print(f"X_test shape:  {tuple(X_test.shape)}")
print(f"y_train shape:  {tuple(y_train.shape)}")
print(f"y_test shape:  {tuple(y_test.shape)}")

In [ ]:
print(f"X_train type:  {X_train.dtype}")
print(f"y_train type:  {y_train[0].dtype}")
print(f"y_train example:  {y_train[0]}")

In [ ]:
import matplotlib.pyplot as plt


n = 10
f, ax = plt.subplots(1, n, figsize=(n * 1.4, 2))
for i in range(n):
    ax[i].imshow(X_train[i], cmap="gray_r")
    ax[i].set_title(int(y_train[i]))
    ax[i].axis("off")
plt.show()

## Transforming the Data

We need to apply a few transformations to this data:

1. We could keep the original tensor shapes `(_, 28, 28)`, but it will be easier to work on tensors of shape `(_, 28²)`, where `_` is the original number of examples.
2. We want to use the $[0, 1]$ range rather than values between $0$ and $255$ for our images. Transform the inputs accordingly.
3. We'll see two ways to define the loss function. The first one requires the classes to be [one-hot encoded](https://en.wikipedia.org/wiki/One-hot).

A few functions that may come in handy:

- [`torch.Tensor.reshape`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)
- [`torch.nn.functional.one_hot`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)

*Apply the first two transformations to `X_train` and `X_test`, and the last one to `y_train` and `y_test`.*

In [ ]:
# Your code here

### Solution

In [ ]:
nb_classes = 10
input_dim = 28 * 28

# We want to "flatten" X so that our network's input is a vector
# of size input_dim
X_train = X_train.reshape(X_train.shape[0], input_dim)
# equivalent syntax
X_train = X_train.reshape(-1, input_dim)

X_test = X_test.reshape(X_test.shape[0], input_dim)
# equivalent syntax
X_test = X_test.reshape(-1, input_dim)

# Transform to [0, 1]
X_train = X_train / 255.0
X_test = X_test / 255.0

# Convert the classes to one-hot vectors
Y_train = nn.functional.one_hot(y_train, nb_classes).float()
Y_test = nn.functional.one_hot(y_test, nb_classes).float()

## Creating a Model

Create a model without a hidden layer that takes an image as input and tries to predict the corresponding class.

Display a summary of this model and explain the numbers you see.

The `LightningModule` below takes care of the loss function, the metric and the optimizer: the model given to it returns logits, cross-entropy accepting class indices as well as one-hot vectors as targets.

In [ ]:
class MNISTClassifier(lightning.LightningModule):
  """Lightning wrapper: cross-entropy on the logits, accuracy as a metric."""

  def __init__(self,
               model: nn.Module,
               optimizer: type[torch.optim.Optimizer] = torch.optim.SGD,
               learning_rate: float = 0.01,
               weight_decay: float = 0.,
               num_classes: int = nb_classes) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # An example input lets Lightning display the shapes of the input and output
    # tensors of each layer in the model summary
    self.example_input_array = torch.zeros(1, input_dim)
    # torchmetrics needs one metric instance per stage
    self.accuracies = nn.ModuleDict({
        f"{stage}_accuracy": torchmetrics.Accuracy(task="multiclass",
                                                   num_classes=num_classes)
        for stage in ("train", "val", "test")
    })

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor],
            stage: str) -> torch.Tensor:
    images, targets = batch
    logits = self(images)
    loss = nn.functional.cross_entropy(logits, targets)
    # The metric, however, needs class indices
    labels = targets.argmax(dim=-1) if targets.dim() > 1 else targets
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, labels)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True,
             prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                batch_index: int) -> torch.Tensor:
    return self._step(batch, "test")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return self.hparams.optimizer(self.parameters(),
                                  lr=self.hparams.learning_rate,
                                  weight_decay=self.hparams.weight_decay)


def get_dataloaders(X: torch.Tensor,
                    Y: torch.Tensor,
                    batch_size: int = 128,
                    validation_split: float = 0.2
                    ) -> tuple[DataLoader, DataLoader]:
  """Split the data between training and validation."""
  train_dataset, val_dataset = random_split(
      TensorDataset(X, Y), [1 - validation_split, validation_split])
  return (DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
          DataLoader(val_dataset, batch_size=batch_size))

In [ ]:
# model = nn.Sequential(???)
# classifier = MNISTClassifier(model)
# print(ModelSummary(classifier, max_depth=-1))

### Solution

In [ ]:
model = nn.Sequential(
    nn.Linear(input_dim, nb_classes))
classifier = MNISTClassifier(model)
print(ModelSummary(classifier, max_depth=-1))

## Training

Train this model on the data.

We'll use:

- 128 as the batch size
- 10 epochs
- 20% of the training set as the validation set

In [ ]:
# Your code here

### Solution

In [ ]:
# The Trainer uses the LightningModule in a complete training loop
train_loader, val_loader = get_dataloaders(X_train, Y_train, batch_size=128)

trainer = lightning.Trainer(max_epochs=10,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="simple"),
                            enable_checkpointing=False)
trainer.fit(classifier, train_loader, val_loader)


def evaluate(trainer: lightning.Trainer,
             classifier: lightning.LightningModule,
             one_hot: bool) -> None:
  test_loader = DataLoader(TensorDataset(X_test,
                                         Y_test if one_hot else y_test),
                           batch_size=128)
  score = trainer.test(classifier, test_loader, verbose=False)[0]
  print(f"Test loss: {score['test_loss']}")
  print(f"Test accuracy: {score['test_accuracy']}")


evaluate(trainer, classifier, True)

Alternative solution that directly uses `y_train` rather than `Y_train` as the cross-entropy target:

In [ ]:
model2 = nn.Sequential(
    nn.Linear(input_dim, nb_classes))
classifier2 = MNISTClassifier(model2)
print(ModelSummary(classifier2, max_depth=-1))

train_loader2, val_loader2 = get_dataloaders(X_train, y_train, batch_size=128)

trainer2 = lightning.Trainer(max_epochs=10,
                             accelerator="auto",
                             devices=1,
                             logger=CSVLogger("logs", name="simple_sparse"),
                             enable_checkpointing=False)
trainer2.fit(classifier2, train_loader2, val_loader2)

evaluate(trainer2, classifier2, one_hot=False)

Solution with 4 hidden layers of size 20, L2 regularization of the parameters, and an orthogonal initialization of the weight matrices

In [ ]:
def dense(in_features: int, out_features: int, activation: bool = True
          ) -> nn.Module:
  layer = nn.Linear(in_features, out_features)
  nn.init.orthogonal_(layer.weight)
  return nn.Sequential(layer, nn.ReLU()) if activation else layer


def build_deep_model() -> nn.Module:
  # The regularization of the weights and biases is handled by the optimizer:
  # AdamW applies weight decay directly to the weights, while an L2 penalty
  # added to the loss would be distorted by Adam's adaptive steps
  return nn.Sequential(
      dense(input_dim, 20),
      dense(20, 20),
      dense(20, 20),
      dense(20, 20),
      dense(20, nb_classes, activation=False))


deep_classifier = MNISTClassifier(build_deep_model(),
                                  optimizer=torch.optim.AdamW,
                                  learning_rate=1e-3,
                                  weight_decay=0.01)
print(ModelSummary(deep_classifier, max_depth=-1))

train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=128)

deep_trainer = lightning.Trainer(max_epochs=10,
                                 accelerator="auto",
                                 devices=1,
                                 logger=CSVLogger("logs", name="deep"),
                                 enable_checkpointing=False)
deep_trainer.fit(deep_classifier, train_loader, val_loader)
evaluate(deep_trainer, deep_classifier, one_hot=False)

## Hyperparameter Search

To find the optimal hyperparameters, you can use the [`optuna`](https://optuna.org/) library. You need to define an objective function that samples the parameters, trains the corresponding model and returns the metric to optimize. Refer to the example on Optuna's home page to define such a function, then use [the TPE Bayesian sampler](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.TPESampler.html), the default one, to find your model's optimal hyperparameters.

In [ ]:
# def objective(trial: optuna.Trial) -> float:
#   ???
# study = optuna.create_study(???)
# study.optimize(???)

### Solution

In [ ]:
EXECUTIONS_PER_TRIAL = 3


def objective(trial: optuna.Trial) -> float:
  units = trial.suggest_int("units", 32, 512, step=32)
  learning_rate = trial.suggest_categorical("learning_rate",
                                            [1e-2, 1e-3, 1e-4])

  # Each configuration is trained several times, the objective being the
  # mean of the validation accuracies obtained
  val_accuracies = []
  for _ in range(EXECUTIONS_PER_TRIAL):
    model = nn.Sequential(nn.Linear(input_dim, units),
                          nn.ReLU(),
                          nn.Linear(units, nb_classes))
    classifier = MNISTClassifier(model,
                                 optimizer=torch.optim.Adam,
                                 learning_rate=learning_rate)
    train_loader, val_loader = get_dataloaders(X_train, y_train,
                                               batch_size=1500)
    trainer = lightning.Trainer(max_epochs=10,
                                accelerator="auto",
                                devices=1,
                                logger=False,
                                enable_checkpointing=False,
                                enable_progress_bar=False,
                                enable_model_summary=False)
    trainer.fit(classifier, train_loader, val_loader)
    val_accuracies.append(
        float(trainer.callback_metrics["val_accuracy"]))
  return sum(val_accuracies) / len(val_accuracies)


study = optuna.create_study(direction="maximize", study_name="mnist")

In [ ]:
study.optimize(objective, n_trials=5)

In [ ]:
print(study.trials_dataframe())
print(f"Best hyperparameters: {study.best_params}")
print(f"Best validation accuracy: {study.best_value}")

## Using TensorBoard to Visualize Metrics

Passing a [`TensorBoardLogger`](https://lightning.ai/docs/pytorch/stable/extensions/generated/lightning.pytorch.loggers.TensorBoardLogger.html) to the `Trainer` in its `logger` argument enables TensorBoard. You can then visualize the training runs directly in Colab with the `tensorboard` extension.

In [ ]:
now = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=3000)

deep_classifier = MNISTClassifier(build_deep_model(),
                                  optimizer=torch.optim.AdamW,
                                  learning_rate=1e-3,
                                  weight_decay=0.01)

deep_trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=TensorBoardLogger("logs", name=f"adamw-{now}"),
    enable_checkpointing=False)
deep_trainer.fit(deep_classifier, train_loader, val_loader)


# Same model but with sgd as the optimizer
deep_classifier = MNISTClassifier(build_deep_model(),
                                  optimizer=torch.optim.SGD,
                                  learning_rate=0.01,
                                  weight_decay=0.01)

deep_trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=TensorBoardLogger("logs", name=f"sgd-{now}"),
    enable_checkpointing=False)
deep_trainer.fit(deep_classifier, train_loader, val_loader)

%reload_ext tensorboard
%tensorboard --logdir logs